# F2 — 03 이어 학습 (3에폭 → 5에폭) · Colab

**무엇**: 지금 1등 모델 **03(경제 사전학습 roberta-large + CRF, 3에폭, dev 0.9251 · test 0.9258)** 의 학습이 끝난 가중치(본체 + 분류층 + CRF)를 그대로 읽어 **2에폭을 더** 학습한다. 에폭을 늘리면 좋아지는지 확인하는 실험.

- 출발: 드라이브 `results/F03_dapt_roberta-large_crf/model/` (처음부터 다시 학습하지 않음)
- 스크립트: `fin_ft_cont.py` = 공용 `fin_ft.py` 복사본 + `--from_fin`(금융 모델 통째로 읽기) 옵션. 공용 파일은 건드리지 않음.
- 설정: 2에폭 · 학습률 1e-5(03은 2e-5에서 0까지 내려간 상태라 작게 다시 시작) · CRF 학습률 1e-4 · 예열 5% · **배치 16(03과 같음)** · 시드 42
- 결과 이름: `F03c_dapt_roberta-large_crf_ep5`. 로그의 "에폭 1·2" = 전체로는 **4·5에폭**.
- 판단: **dev 로만** 03의 3에폭 dev 0.9251 과 비교. test 는 마지막에 한 번 나오는 것만 기록(설정을 test 보고 바꾸지 않기).
- 채점은 공통(`fin_score.py`) 그대로: dev · test · test_2000 · 크롤링 350(Fable). 후처리(조사 자르기 등)는 아직 안 넣음.

## 런타임
- **G4(RTX PRO 6000) 권장** — 03s(large)가 G4에서 3에폭 49분 → 2에폭 약 30~40분(추정). A100이면 약 1.5시간(03 실측 3에폭 141분에서 환산).
- RAM 표준. 마지막 셀이 런타임 반납.

In [1]:
# 1. GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다'
p = torch.cuda.get_device_properties(0); print(p.name, f'{p.total_memory/2**30:.0f}GB', 'bf16', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), 'bf16 미지원 GPU(T4) → A100 으로 바꾸세요'

NVIDIA RTX PRO 6000 Blackwell Server Edition 95GB bf16 True


In [2]:
# 2. 드라이브 연결 · 폴더 찾기 (NER_라벨_통합_v1.2/finetune)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, time
ROOT = '/content/drive/MyDrive/NER_라벨_통합_v1.2/finetune'   # 다르면 여기만 수정
if not os.path.exists(f'{ROOT}/fin_ft.py'):
    found = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/fin_ft.py', recursive=True)
             + glob.glob('/content/drive/Shareddrives/*/**/fin_ft.py', recursive=True)]
    print('찾은 위치:', found); assert found, 'fin_ft.py 를 못 찾음 → 공유 폴더를 내 드라이브에 바로가기로 추가했는지 확인'
    ROOT = found[0]
print('ROOT =', ROOT)
for f in ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'test_2000.jsonl']:
    print(f, sum(1 for _ in open(f'{ROOT}/data/{f}', encoding='utf-8')), '문장')
print('모델:', sorted(os.listdir(f'{ROOT}/models')))

Mounted at /content/drive
ROOT = /content/drive/MyDrive/NER_라벨_통합_v1.2/finetune
train.jsonl 279925 문장
dev.jsonl 10040 문장
test.jsonl 10035 문장
test_2000.jsonl 2000 문장
모델: ['01_stage1-1200k_roberta-base', '02_dapt_roberta-base', '03_dapt_roberta-large', '04_dapt_mmbert-ep1', '05_dapt_mmbert-ep2', '06_dapt-stage1-5000k_roberta-base', '07_dapt-stage1-5000k_roberta-large', '08_dapt-stage1-5000k_mmbert', '09_gptoss_dapt-lora']


In [3]:
# 3. 라이브러리
!pip install -q pytorch-crf kiwipiepy
import transformers; print('transformers', transformers.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
transformers 5.20.2


In [ ]:
# 4. 이어 학습 설정 — 03(3에폭 끝)을 2에폭 더
SRC = f'{ROOT}/results/F03_dapt_roberta-large_crf/model'
assert os.path.exists(f'{SRC}/model.safetensors') and os.path.exists(f'{SRC}/fin_ner_config.json'), f'03 모델이 없습니다: {SRC}'
assert os.path.exists(f'{ROOT}/fin_ft_cont.py'), 'fin_ft_cont.py 가 드라이브 finetune/ 에 없습니다'
EXP = 'F03c_dapt_roberta-large_crf_ep5'
HEAD, PRETOK = 'crf', 'none'                    # 03 과 같아야 함(다르면 스크립트가 멈춤)
LR, CRF_LR, EPOCHS, WARMUP = 1e-5, 1e-4, 2, 0.05
L = '/content/local'; os.makedirs(L, exist_ok=True)
for src, dst in [(SRC, f'{L}/model'), (f'{ROOT}/data', f'{L}/data')]:
    t = time.time(); shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst); print('복사', dst, f'{time.time()-t:.0f}초')
INIT = f'{L}/model'
print(json.load(open(f'{INIT}/fin_ner_config.json', encoding='utf-8')))
EXTRA = f'{ROOT}/data/crawl350_fable.jsonl'
print('이어 학습', EXP, '| lr', LR, '| crf_lr', CRF_LR, '| 추가 에폭', EPOCHS, '| 추가 채점', os.path.exists(EXTRA))

복사 /content/local/model 19초
복사 /content/local/data 1초
{'labels': ['O', 'B-DATE', 'I-DATE', 'B-FIN_ORG', 'I-FIN_ORG', 'B-FIN_PRODUCT', 'I-FIN_PRODUCT', 'B-INDICATOR', 'I-INDICATOR', 'B-MONEY', 'I-MONEY', 'B-ORG', 'I-ORG', 'B-POLICY', 'I-POLICY', 'B-RATE', 'I-RATE'], 'tags': ['DATE', 'FIN_ORG', 'FIN_PRODUCT', 'INDICATOR', 'MONEY', 'ORG', 'POLICY', 'RATE'], 'head': 'crf', 'pretok': 'none', 'init': '/content/local/model', 'max_len': 256}
이어 학습 F03c_dapt_roberta-large_crf_ep5 | lr 1e-05 | crf_lr 0.0001 | 추가 에폭 2 | 추가 채점 True


In [ ]:
# 5. 이어 학습 + 채점 (끊기면 1~5번 다시 실행 → 드라이브 체크포인트부터 이어서)
os.environ['TQDM_MININTERVAL'] = '60'   # 진행 막대 60초에 한 번
OUT = '/content/out'
EX = f'--eval_file "{EXTRA}"' if os.path.exists(EXTRA) else ''
!cd /content && python "{ROOT}/fin_ft_cont.py" --init {INIT} --from_fin --exp {EXP} --data {L}/data --head {HEAD} --pretok {PRETOK} --lr {LR} --crf_lr {CRF_LR} --warmup {WARMUP} --epochs {EPOCHS} --out_root {OUT} --mirror "{ROOT}/results/_ckpt" --batch 16 --workers 8 --save_steps 6000 {EX}

In [ ]:
# 6. 결과 확인 → 모델·결과·오답 파일을 드라이브 results/EXP 에 올리기
r = json.load(open(f'{OUT}/results/{EXP}.json', encoding='utf-8'))
print('에폭별 dev (1·2 = 전체 4·5에폭, 03의 3에폭 dev 0.9251):', r['history'])
for k in ['dev', 'test', 'test_2000', 'extra']:
    if k in r: print(f"{k:>9}: F1 {r[k]['ALL']['f1']} (P {r[k]['ALL']['precision']} R {r[k]['ALL']['recall']}) · macro {r[k]['MACRO_F1']} · 오답 {r[k]['error_types']}")
print('test 태그별 F1:', {t: r['test'][t]['f1'] for t in r['tags']})
print('학습', round(r['train_sec']/60, 1), '분 ·', r['gpu'])
UP = f'{ROOT}/results/{EXP}'; os.makedirs(UP, exist_ok=True)
shutil.copytree(f'{OUT}/models/{EXP}', f'{UP}/model', dirs_exist_ok=True)
shutil.copy(f'{OUT}/results/{EXP}.json', UP)
for f in glob.glob(f'{OUT}/preds/{EXP}_*.jsonl'): shutil.copy(f, UP)
same = all(os.path.getsize(f'{OUT}/models/{EXP}/{f}') == os.path.getsize(f'{UP}/model/{f}') for f in os.listdir(f'{OUT}/models/{EXP}'))
print('드라이브 업로드', UP, '· 크기 일치', same, '·', sorted(os.listdir(UP)))
OK = bool(r.get('done')) and same

In [ ]:
# 7. 런타임 반납(세션 종료) — 6번이 OK일 때만. 노션 분배표에 dev·test F1 을 적어 주세요
from google.colab import runtime
if OK: print('반납'); runtime.unassign()
else: print('업로드 확인 실패 → 반납하지 않음. 출력을 공유해 주세요')